In [1]:
import torch

from rlaopt.linalg import IdentityConfig, LinSys, NystromConfig
from rlaopt.solvers import PCG, PCGConfig

In [2]:
torch.set_default_dtype(torch.float64)

In [3]:
n = 10000
eigvals = torch.arange(1, n + 1) ** -2.0
reg = 1e-3

U = torch.randn(n, n)
U = torch.linalg.qr(U).Q

A = U @ torch.diag(eigvals) @ U.T
B = torch.randn(n, 10)

In [4]:
lin_sys = LinSys(A, B, reg)
lin_sys.cuda()

LinSys()

In [5]:
preconditioner_config_identity = IdentityConfig()
preconditioner_config_nystrom = NystromConfig(rank=100, base_damping=reg)

In [6]:
solver_config = PCGConfig(
    tol=1e-8,
    preconditioner_config=preconditioner_config_nystrom,
)

In [7]:
solver = PCG(solver_config, lin_sys)
params = lin_sys.w.clone()
state = solver.init_state(params)

In [8]:
max_iters = 100

for i in range(max_iters):
    params, state = solver.step(params, state)
    print(f"Iteration {i + 1}, residual norm: {state.res_norm}")

Iteration 1, residual norm: tensor([1.8575, 1.9343, 1.5863, 1.5807, 1.6284, 1.7182, 1.8928, 1.6065, 2.1205,
        1.6313], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 2, residual norm: tensor([0.1523, 0.1457, 0.1052, 0.1023, 0.1125, 0.1523, 0.1214, 0.1116, 0.1756,
        0.1242], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 3, residual norm: tensor([0.0124, 0.0088, 0.0082, 0.0094, 0.0072, 0.0110, 0.0101, 0.0073, 0.0129,
        0.0072], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 4, residual norm: tensor([0.0006, 0.0005, 0.0004, 0.0005, 0.0004, 0.0005, 0.0006, 0.0004, 0.0007,
        0.0004], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 5, residual norm: tensor([3.2857e-05, 3.0891e-05, 2.0265e-05, 2.6319e-05, 2.0265e-05, 2.1610e-05,
        2.6655e-05, 2.6594e-05, 3.7163e-05, 1.7759e-05], device='cuda:0',
       grad_fn=<LinalgVectorNormBackward0>)
Iteration 6, residual norm: tensor([1.6919e-06, 1.2141e-06